In [5]:
import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

def parse_paths():
    if len(sys.argv) > 2 and not sys.argv[1].startswith('-f'):
        return Path(sys.argv[1]), Path(sys.argv[2])
    cands = [
        Path('./dataset/public'),
        Path('/kaggle/input/datasets/devansh66/logical-bundle'),
        Path('/kaggle/input/logical-bundle'),
        Path('/kaggle/input'),
        Path('.')
    ]
    pub = next((c for c in cands if (c / 'tasks.json').exists()), Path('./dataset/public'))
    out = Path('./working/submission.csv')
    os.makedirs(out.parent, exist_ok=True)
    return pub, out

PUB, OUT = parse_paths()
catalog = json.load(open(PUB / 'catalog.json', encoding='utf-8'))
tasks = json.load(open(PUB / 'tasks.json', encoding='utf-8'))

item_map = {int(c['item_id']): c for c in catalog}
catalog_ids = np.array([int(c['item_id']) for c in catalog])
id_to_idx = {int(item_id): idx for idx, item_id in enumerate(catalog_ids)}

prices = np.array([float(c['price']) for c in catalog], dtype=np.float64)

categories = sorted(list({c['category'] for c in catalog}))
colors = sorted(list({c['color'] for c in catalog}))
brands = sorted(list({c['brand'] for c in catalog}))

cat_indices = np.array([categories.index(c['category']) for c in catalog])
color_indices = np.array([colors.index(c['color']) for c in catalog])
brand_indices = np.array([brands.index(c['brand']) for c in catalog])

cat_to_catalog_indices = {k: np.where(cat_indices == k)[0] for k in range(len(categories))}

category_ranks = np.zeros(len(catalog), dtype=np.float64)
for k, idxs in cat_to_catalog_indices.items():
    sorted_order = idxs[np.argsort(-prices[idxs])]
    category_ranks[sorted_order] = np.arange(len(sorted_order)) / max(len(sorted_order) - 1, 1)

N_FEATURES = 22

def extract_candidate_features(cand_indices, input_indices):
    cand_prices = prices[cand_indices]
    inp_prices = prices[input_indices]
    
    p_min = np.min(inp_prices)
    p_max = np.max(inp_prices)
    p_mean = np.mean(inp_prices)
    p_sum = np.sum(inp_prices)
    
    scale = 500.0
    
    idx_min = input_indices[int(np.argmin(inp_prices))]
    idx_max = input_indices[int(np.argmax(inp_prices))]
    idx_first = input_indices[0]
    idx_last = input_indices[-1]
    
    key_items = {'lo': idx_min, 'hi': idx_max, 'f': idx_first, 'l': idx_last}
    
    F = np.empty((len(cand_indices), N_FEATURES), dtype=np.float64)
    F[:, 0] = category_ranks[cand_indices]
    F[:, 1] = cand_prices / scale
    F[:, 2] = -np.abs(cand_prices - p_min) / scale
    F[:, 3] = -np.abs(cand_prices - p_max) / scale
    F[:, 4] = -np.abs(cand_prices - p_mean) / scale
    F[:, 5] = -np.abs(cand_prices - p_sum) / scale
    F[:, 6] = (cand_prices > p_max).astype(np.float64)
    F[:, 7] = (cand_prices < p_min).astype(np.float64)
    
    col_ctr = 8
    for key in ('lo', 'hi', 'f', 'l'):
        target_idx = key_items[key]
        F[:, col_ctr] = (color_indices[cand_indices] == color_indices[target_idx]).astype(np.float64)
        col_ctr += 1
        F[:, col_ctr] = (brand_indices[cand_indices] == brand_indices[target_idx]).astype(np.float64)
        col_ctr += 1
        
    mode_color = Counter(color_indices[input_indices].tolist()).most_common(1)[0][0]
    mode_brand = Counter(brand_indices[input_indices].tolist()).most_common(1)[0][0]
    
    F[:, col_ctr] = (color_indices[cand_indices] == mode_color).astype(np.float64)
    col_ctr += 1
    F[:, col_ctr] = (brand_indices[cand_indices] == mode_brand).astype(np.float64)
    col_ctr += 1
    F[:, col_ctr] = np.isin(color_indices[cand_indices], color_indices[input_indices]).astype(np.float64)
    col_ctr += 1
    F[:, col_ctr] = np.isin(brand_indices[cand_indices], brand_indices[input_indices]).astype(np.float64)
    col_ctr += 1
    F[:, col_ctr] = 1.0
    
    return F

def fit_ridge_model(X, y, alpha=1.0):
    A = X.T @ X + alpha * np.eye(X.shape[1])
    return np.linalg.solve(A, X.T @ y)

submission_rows = []

for task_id, task in tasks.items():
    train_examples = [
        (
            np.array([id_to_idx[int(i)] for i in e['input']]),
            sorted([int(o) for o in e['output']], key=lambda i: (cat_indices[id_to_idx[i]], i))
        )
        for e in task['train']
    ]
    
    target_cats = [cat_indices[id_to_idx[o]] for o in train_examples[0][1]]
    slot_weights = {}
    
    for slot_idx, slot_cat in enumerate(target_cats):
        cand_indices = cat_to_catalog_indices[slot_cat]
        X_list, y_list = [], []
        
        for inp_idx, out_ids in train_examples:
            if slot_idx >= len(out_ids):
                continue
            target_item_idx = id_to_idx[out_ids[slot_idx]]
            F = extract_candidate_features(cand_indices, inp_idx)
            y = (cand_indices == target_item_idx).astype(np.float64)
            if y.sum() == 0:
                continue
            X_list.append(F)
            y_list.append(y)
            
        if not X_list:
            continue
            
        X_task = np.vstack(X_list)
        y_task = np.concatenate(y_list)
        w = fit_ridge_model(X_task, y_task)
        slot_weights[slot_idx] = w
        
    test_input = [id_to_idx[int(i)] for i in task['test'][0]['input']]
    predicted_item_ids = []
    
    for slot_idx, slot_cat in enumerate(target_cats):
        if slot_idx not in slot_weights:
            continue
        w = slot_weights[slot_idx]
        cand_indices = cat_to_catalog_indices[slot_cat]
        F_test = extract_candidate_features(cand_indices, test_input)
        scores = F_test @ w
        ranked_candidates = cand_indices[np.argsort(-scores)]
        
        for cand_idx in ranked_candidates:
            item_id = int(catalog_ids[cand_idx])
            if item_id not in predicted_item_ids and item_id not in [int(i) for i in task['test'][0]['input']]:
                predicted_item_ids.append(item_id)
                break
                
    if not predicted_item_ids:
        predicted_item_ids = [int(catalog_ids[0])]
        
    submission_rows.append((task_id, ' '.join(str(x) for x in predicted_item_ids)))

sub_df = pd.DataFrame(submission_rows, columns=['task_id', 'recommended_item_ids'])
ss_df = pd.read_csv(PUB / 'sample_submission.csv')
sub_df = ss_df[['task_id']].merge(sub_df, on='task_id', how='left')
sub_df['recommended_item_ids'] = sub_df['recommended_item_ids'].fillna(str(catalog_ids[0]))

sub_df.to_csv(OUT, index=False)

/tmp/ipykernel_58/4066629821.py:103: RuntimeWarning: overflow encountered in matmul
  A = X.T @ X + alpha * np.eye(X.shape[1])
/tmp/ipykernel_58/4066629821.py:103: RuntimeWarning: invalid value encountered in matmul
  A = X.T @ X + alpha * np.eye(X.shape[1])
/tmp/ipykernel_58/4066629821.py:104: RuntimeWarning: invalid value encountered in matmul
  return np.linalg.solve(A, X.T @ y)
